In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
del os.environ["BIRDDOG_USE_LOCAL_CACHE"]
if os.environ.get("BIRDDOG_USE_LOCAL_CACHE"):
    print("using local cache")
else:
    print("using aws cache")

using aws cache


In [19]:
import json
from birddog.runtime import ArchiveWatcher
from birddog.user import _watcher_cache_path
from birddog.cache import load_cached_object
from birddog.store import DynamoDBKeyValueStore

In [20]:
store = DynamoDBKeyValueStore()

In [4]:
email = "jberland@jewishgen.org"

In [21]:
watchlists = store.get_all(f"wl:{email}")

In [23]:
watchlists[0]

('AGAD-_',
 '{"cutoff_date": "2025-12-01T00:00:00Z", "last_checked_date": "2026-07-17T13:32:51Z"}')

In [5]:
def get_watcher_data(archive, subarchive="_"):
    path = _watcher_cache_path(email, archive, subarchive)
    return load_cached_object(path)

In [6]:
data = get_watcher_data("AGAD")

In [7]:
aw = ArchiveWatcher.load(get_watcher_data("AGAD"))

2026-07-17 08:40:50,088 [INFO] ArchiveWatcher init (runtime=None)


In [ ]:
#with open("var/AGAD.json", "w") as f:
#    f.write(json.dumps(data))

In [25]:
def scan_wl_data(watcher):
    for k,v in watcher.unresolved.items():
        if v.get("last_resolved") and v.get("modified"):
            if v.get("last_resolved") > v.get("modified"):
                print("mismatch:  ", k,v.get("last_resolved"),v.get("modified"))

In [34]:
def check(archive, subarchive="_"):
    watcher_data = get_watcher_data(archive, subarchive)
    with open(f"./var/juliana/{archive}-{subarchive}.json", "w") as f:
        f.write(json.dumps(watcher_data))
    aw = ArchiveWatcher.load(get_watcher_data(archive, subarchive))
    scan_wl_data(aw)

In [35]:
check("AGAD")

2026-07-17 08:57:23,131 [INFO] ArchiveWatcher init (runtime=None)


In [36]:
check("CDIAK")

2026-07-17 08:57:25,407 [INFO] ArchiveWatcher init (runtime=None)


In [37]:
for item in watchlists:
    archive, subarchive = item[0].split("-")
    print(f"checking {archive}-{subarchive}")
    check(archive, subarchive)

checking AGAD-_
2026-07-17 08:57:28,273 [INFO] ArchiveWatcher init (runtime=None)
checking ANK-_
2026-07-17 08:57:28,445 [INFO] ArchiveWatcher init (runtime=None)
checking AVPRI-_
2026-07-17 08:57:28,616 [INFO] ArchiveWatcher init (runtime=None)
checking CDIAK-_
2026-07-17 08:57:29,089 [INFO] ArchiveWatcher init (runtime=None)
checking CSAMM-_
2026-07-17 08:57:29,271 [INFO] ArchiveWatcher init (runtime=None)
checking DAARK-D
2026-07-17 08:57:29,500 [INFO] ArchiveWatcher init (runtime=None)
checking DACHGO-D
2026-07-17 08:57:29,714 [INFO] ArchiveWatcher init (runtime=None)
checking DACHGO-R
2026-07-17 08:57:30,246 [INFO] ArchiveWatcher init (runtime=None)
checking DACHKO-D
2026-07-17 08:57:31,684 [INFO] ArchiveWatcher init (runtime=None)
checking DACHKO-R
2026-07-17 08:57:32,051 [INFO] ArchiveWatcher init (runtime=None)
checking DACHVO-D
2026-07-17 08:57:32,258 [INFO] ArchiveWatcher init (runtime=None)
checking DACHVO-R
2026-07-17 08:57:32,434 [INFO] ArchiveWatcher init (runtime=None)
c